# Olist E-Commerce - 03. Customer & Logistics Analysis
## Geographic Demographics, Repeat Purchase Cohorts & Delivery SLA vs Customer Satisfaction

### Objective
This notebook analyzes buyer demographics, regional concentration across Brazilian states, repeat customer rates, and the empirical correlation between delivery transit speed and review scores.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path('..').resolve()
DATA_CLEAN = PROJECT_ROOT / 'data' / 'processed' / 'cleaned'

customers = pd.read_csv(DATA_CLEAN / 'olist_customers_dataset.csv')
orders = pd.read_csv(DATA_CLEAN / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'
])
reviews = pd.read_csv(DATA_CLEAN / 'olist_order_reviews_dataset.csv')
sellers = pd.read_csv(DATA_CLEAN / 'olist_sellers_dataset.csv')
order_items = pd.read_csv(DATA_CLEAN / 'olist_order_items_dataset.csv')

print("Loaded all datasets successfully.")


### 1. Customer Geographic Concentration (State & City Level)


In [ ]:
df_cust_orders = orders.merge(customers, on='customer_id', how='inner')
state_orders = df_cust_orders['customer_state'].value_counts()
state_pct = (state_orders / len(df_cust_orders) * 100).round(2)

state_summary = pd.DataFrame({'Orders': state_orders, 'Percentage': state_pct}).head(10)
print("Top 10 States by Order Volume:")
display(state_summary)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=state_summary.index, y=state_summary['Orders'], palette='Blues_r', ax=ax)
ax.set_title('Top 10 Customer States by Order Volume', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Orders')
for i, v in enumerate(state_summary['Orders']):
    pct_val = state_summary['Percentage'].iloc[i]
    ax.text(i, v + 800, f'{v:,}\n({pct_val}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


### 2. Repeat Customer Rate Analysis
Assessing customer loyalty and multi-order frequency.


In [ ]:
cust_freq = customers.groupby('customer_unique_id').agg(
    Order_Count=('customer_id', 'count')
).reset_index()

repeat_custs = cust_freq[cust_freq['Order_Count'] > 1]
repeat_rate = (len(repeat_custs) / len(cust_freq)) * 100

print(f"Total Unique Customers: {len(cust_freq):,}")
print(f"Repeat Customers (2+ orders): {len(repeat_custs):,} ({repeat_rate:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
freq_dist = cust_freq['Order_Count'].value_counts().sort_index().head(5)
ax.bar(freq_dist.index.astype(str), freq_dist.values, color=['#0084FF', '#FF8A00', '#00C49F', '#7952B3', '#AAAAAA'])
ax.set_title('Customer Order Frequency Distribution', fontweight='bold')
ax.set_xlabel('Number of Orders Placed')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(freq_dist.values):
    ax.text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


### 3. Delivery SLA & Transit Performance
Examining transit days vs. promised estimated delivery dates for completed deliveries.


In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].dropna(subset=['order_delivered_customer_date']).copy()

# Calculate delivery days
delivered['actual_delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
delivered['estimated_delivery_days'] = (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
delivered['delivery_delay_days'] = (delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)

delivered['is_on_time'] = delivered['order_delivered_customer_date'] <= delivered['order_estimated_delivery_date']

on_time_rate = delivered['is_on_time'].mean() * 100

print(f"Delivered Orders Audited: {len(delivered):,}")
print(f"Average Actual Transit Days: {delivered['actual_delivery_days'].mean():.2f} days")
print(f"Average Estimated SLA: {delivered['estimated_delivery_days'].mean():.2f} days")
print(f"On-Time Delivery Rate: {on_time_rate:.2f}%")


### 4. Correlation Between Delivery Speed and Customer Satisfaction (Review Scores)
Analyzing how delivery delays destroy review ratings.


In [ ]:
df_del_rev = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')

# Categorize delivery outcome
def categorize_delivery(row):
    delay = row['delivery_delay_days']
    if delay <= -5:
        return 'Delivered > 5 Days Early'
    elif delay <= 0:
        return 'Delivered 0-5 Days Early'
    elif delay <= 5:
        return 'Delayed 1-5 Days'
    else:
        return 'Delayed > 5 Days'

df_del_rev['delivery_category'] = df_del_rev.apply(categorize_delivery, axis=1)

sla_review_summary = df_del_rev.groupby('delivery_category').agg(
    Orders=('order_id', 'count'),
    Avg_Review_Score=('review_score', 'mean'),
    Pct_5_Star=('review_score', lambda x: (x == 5).mean() * 100),
    Pct_1_Star=('review_score', lambda x: (x == 1).mean() * 100)
).reindex([
    'Delivered > 5 Days Early',
    'Delivered 0-5 Days Early',
    'Delayed 1-5 Days',
    'Delayed > 5 Days'
]).reset_index()

display(sla_review_summary.round(2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=sla_review_summary, x='delivery_category', y='Avg_Review_Score', palette='RdYlGn_r', ax=ax1)
ax1.set_title('Average Review Score by Delivery Timeliness', fontweight='bold')
ax1.set_ylabel('Mean Rating (1-5 Stars)')
ax1.set_ylim(1, 5)
ax1.tick_params(axis='x', rotation=20)
for i, v in enumerate(sla_review_summary['Avg_Review_Score']):
    ax1.text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

sns.barplot(data=sla_review_summary, x='delivery_category', y='Pct_1_Star', palette='Reds', ax=ax2)
ax2.set_title('Share of 1-Star Bad Reviews by Timeliness', fontweight='bold')
ax2.set_ylabel('% 1-Star Reviews')
ax2.tick_params(axis='x', rotation=20)
for i, v in enumerate(sla_review_summary['Pct_1_Star']):
    ax2.text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
